# Cell Image Classifier — MSC vs Fibroblast / Differentiated vs Undifferentiated
Transfer-learning pipeline (PyTorch, EfficientNet/ResNet) for microscopy images.

**Before you start:** `Runtime → Change runtime type → GPU (T4)`.

## How to organise your images (Google Drive)
```
cell_data/
├── MSC/                    ← one folder per class (any names: MSC, Fibroblast, Differentiated, Undifferentiated…)
│   ├── donor1/  img001.tif, img002.tif …
│   ├── donor2/  …
│   └── donor3/  …
└── Fibroblast/
    ├── lineA/  …
    └── lineB/  …
```
The **second-level folder = experiment / donor / culture batch**. The notebook keeps every group entirely in
*either* train, validation or test, so the model is tested on cultures it has never seen. Without this, images
from the same flask leak into the test set and accuracy looks falsely perfect.
If you have no subfolders, images go directly under the class folder and a random split is used (with a warning).

Supported formats: .tif/.tiff (8/16-bit), .png, .jpg, .bmp.

In [ ]:
#@title 1. Settings — edit these
DATA_DIR      = "/content/drive/MyDrive/cell_data"   # folder with one subfolder per class
MODEL_NAME    = "efficientnet_b0"   # options: efficientnet_b0, resnet50, convnext_tiny
TILE_SIZE     = 512     # crop large micrographs into tiles of this size (pixels) before resizing
INPUT_SIZE    = 224     # network input size
TILES_PER_IMG = 8       # random tiles per image per epoch (training)
EPOCHS        = 25
BATCH_SIZE    = 32
LR            = 3e-4
SEED          = 42
VAL_FRAC, TEST_FRAC = 0.15, 0.20
OUT_DIR       = "/content/drive/MyDrive/cell_model_output"

In [ ]:
#@title 2. Mount Drive & install
from google.colab import drive
drive.mount('/content/drive')
!pip -q install timm tifffile grad-cam

In [ ]:
#@title 3. Imports & reproducibility
import os, glob, random, json, math, numpy as np, torch, torch.nn as nn, timm, tifffile
from PIL import Image
from collections import Counter, defaultdict
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             ConfusionMatrixDisplay, roc_curve)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
#@title 4. Find images and build a group-aware split
EXTS = (".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp")
classes = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
assert len(classes) >= 2, f"Need ≥2 class folders in {DATA_DIR}, found {classes}"
cls2idx = {c: i for i, c in enumerate(classes)}

records = []   # (path, label, group)
for c in classes:
    for p in glob.glob(os.path.join(DATA_DIR, c, "**", "*"), recursive=True):
        if p.lower().endswith(EXTS):
            rel = os.path.relpath(p, os.path.join(DATA_DIR, c)).split(os.sep)
            group = f"{c}/{rel[0]}" if len(rel) > 1 else None
            records.append((p, cls2idx[c], group))

print("Classes:", classes)
print("Images per class:", {c: sum(r[1]==i for r in records) for c, i in cls2idx.items()})

has_groups = all(r[2] is not None for r in records)
if not has_groups:
    print("⚠️  No experiment/donor subfolders found → random image-level split. "
          "Results may be over-optimistic if images come from the same culture.")
    records = [(p, y, f"img::{p}") for p, y, _ in records]

# split groups per class so every class appears in each split
split = {}
for i, c in enumerate(classes):
    groups = sorted({g for _, y, g in records if y == i})
    random.Random(SEED).shuffle(groups)
    n = len(groups)
    n_test = max(1, round(n * TEST_FRAC)); n_val = max(1, round(n * VAL_FRAC))
    if n < 3:
        raise ValueError(f"Class '{c}' has only {n} group(s). Need ≥3 donors/experiments "
                         "(or remove subfolders to use a random split).")
    for g in groups[:n_test]: split[g] = "test"
    for g in groups[n_test:n_test+n_val]: split[g] = "val"
    for g in groups[n_test+n_val:]: split[g] = "train"

sets = defaultdict(list)
for p, y, g in records: sets[split[g]].append((p, y))
for k in ["train", "val", "test"]:
    print(f"{k:5s}: {len(sets[k]):5d} images  {dict(Counter(classes[y] for _, y in sets[k]))}")
json.dump({"classes": classes, "split": split}, open(f"{OUT_DIR}/split.json", "w"), indent=1)

In [ ]:
#@title 5. Image loading (handles 16-bit TIFF, grayscale, fluorescence)
def load_image(path):
    if path.lower().endswith((".tif", ".tiff")):
        a = tifffile.imread(path)
        a = np.squeeze(a)
        if a.ndim == 3 and a.shape[0] in (2, 3, 4) and a.shape[-1] not in (3, 4):
            a = np.moveaxis(a, 0, -1)          # channels-first → channels-last
        if a.ndim == 3 and a.shape[-1] > 3: a = a[..., :3]
        if a.ndim == 3 and a.shape[-1] == 2: a = np.concatenate([a, a[..., :1]], -1)
    else:
        a = np.array(Image.open(path))
    a = a.astype(np.float32)
    lo, hi = np.percentile(a, (0.5, 99.5))       # robust contrast normalisation
    a = np.clip((a - lo) / (hi - lo + 1e-6), 0, 1)
    if a.ndim == 2: a = np.stack([a]*3, -1)
    if a.shape[-1] == 4: a = a[..., :3]
    return Image.fromarray((a * 255).astype(np.uint8))

def tiles_of(img, size):
    W, H = img.size
    if W <= size or H <= size: return [img]
    xs = list(range(0, W - size + 1, size)); ys = list(range(0, H - size + 1, size))
    return [img.crop((x, y, x+size, y+size)) for y in ys for x in xs]

# quick look
fig, ax = plt.subplots(1, len(classes), figsize=(4*len(classes), 4))
for i, c in enumerate(classes):
    p = next(p for p, y in sets["train"] if y == i)
    ax[i].imshow(load_image(p)); ax[i].set_title(c); ax[i].axis("off")
plt.show()

In [ ]:
#@title 6. Datasets & augmentation
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = T.Compose([
    T.RandomCrop(TILE_SIZE, pad_if_needed=True, padding_mode="reflect"),
    T.Resize(INPUT_SIZE),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomApply([T.RandomRotation(90)], p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.RandomApply([T.GaussianBlur(3)], p=0.2),
    T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize((INPUT_SIZE, INPUT_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

class TrainDS(Dataset):
    "Each item = a random tile from an image (TILES_PER_IMG draws per image per epoch)."
    def __init__(self, items): self.items = items; self.cache = {}
    def __len__(self): return len(self.items) * TILES_PER_IMG
    def __getitem__(self, i):
        p, y = self.items[i // TILES_PER_IMG]
        if p not in self.cache: self.cache[p] = load_image(p)
        return train_tf(self.cache[p]), y

train_ds = TrainDS(sets["train"])
# class-balanced sampling
counts = Counter(y for _, y in sets["train"])
weights = [1.0 / counts[sets["train"][i // TILES_PER_IMG][1]] for i in range(len(train_ds))]
sampler = torch.utils.data.WeightedRandomSampler(weights, len(weights), replacement=True)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)

In [ ]:
#@title 7. Model
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=len(classes)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS*len(train_dl))
crit = nn.CrossEntropyLoss(label_smoothing=0.05)
scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

@torch.no_grad()
def predict_image(path, m=None):
    "Average softmax over all tiles (+ flip TTA) → image-level probabilities."
    m = m or model; m.eval()
    tiles = tiles_of(load_image(path), TILE_SIZE)
    x = torch.stack([eval_tf(t) for t in tiles]).to(device)
    p = (m(x).softmax(1) + m(torch.flip(x, [3])).softmax(1)) / 2
    return p.mean(0).cpu().numpy()

def evaluate(items):
    probs = np.array([predict_image(p) for p, _ in items]); ys = np.array([y for _, y in items])
    return ys, probs

In [ ]:
#@title 8. Train (keeps the best model by validation accuracy, then balanced loss)
best, history = (-1, 1e9), []
for ep in range(1, EPOCHS + 1):
    model.train(); tot = 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
            loss = crit(model(x), y)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item() * len(y)
    ys, pr = evaluate(sets["val"])
    acc = (pr.argmax(1) == ys).mean()
    vloss = -np.mean(np.log(pr[np.arange(len(ys)), ys] + 1e-9))
    history.append((tot / len(train_ds), vloss, acc))
    print(f"epoch {ep:2d}  train_loss {tot/len(train_ds):.3f}  val_loss {vloss:.3f}  val_acc {acc:.3f}")
    if (acc, -vloss) > (best[0], -best[1]):
        best = (acc, vloss); torch.save(model.state_dict(), f"{OUT_DIR}/best_model.pt")

h = np.array(history)
plt.plot(h[:, 0], label="train loss"); plt.plot(h[:, 1], label="val loss"); plt.plot(h[:, 2], label="val acc")
plt.legend(); plt.xlabel("epoch"); plt.show()

In [ ]:
#@title 9. Final evaluation on the held-out TEST set (unseen donors/experiments)
model.load_state_dict(torch.load(f"{OUT_DIR}/best_model.pt", map_location=device))
ys, pr = evaluate(sets["test"]); pred = pr.argmax(1)
print(classification_report(ys, pred, target_names=classes, digits=3))
ConfusionMatrixDisplay(confusion_matrix(ys, pred), display_labels=classes).plot(cmap="Blues"); plt.show()

if len(classes) == 2:
    auc = roc_auc_score(ys, pr[:, 1]); fpr, tpr, _ = roc_curve(ys, pr[:, 1])
    plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}"); plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("1 - specificity"); plt.ylabel("sensitivity"); plt.legend(); plt.show()

# 95% bootstrap CI for accuracy
rng = np.random.default_rng(SEED)
boots = [(pred[i] == ys[i]).mean() for i in (rng.integers(0, len(ys), len(ys)) for _ in range(2000))]
print(f"Test accuracy {np.mean(pred == ys):.3f}  (95% CI {np.percentile(boots, 2.5):.3f}–{np.percentile(boots, 97.5):.3f}, n={len(ys)})")
if np.mean(pred == ys) == 1.0:
    print("⚠️  100% on test: check for leakage (same culture in train & test, labels visible in image, "
          "different microscope/settings per class) before trusting this.")

In [ ]:
#@title 10. Grad-CAM — what part of the image drives the decision?
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
target_layer = (model.conv_head if hasattr(model, "conv_head") else
                model.layer4[-1] if hasattr(model, "layer4") else model.stages[-1])
cam = GradCAM(model=model, target_layers=[target_layer])
model.eval()
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for k, (p, y) in enumerate(random.Random(1).sample(sets["test"], min(4, len(sets["test"])))):
    tile = tiles_of(load_image(p), TILE_SIZE)[0]
    x = eval_tf(tile).unsqueeze(0).to(device)
    g = cam(input_tensor=x)[0]
    rgb = np.array(tile.resize((INPUT_SIZE, INPUT_SIZE))) / 255.0
    pr_ = predict_image(p)
    ax[0, k].imshow(rgb); ax[0, k].set_title(f"true: {classes[y]}"); ax[0, k].axis("off")
    ax[1, k].imshow(show_cam_on_image(rgb.astype(np.float32), g, use_rgb=True))
    ax[1, k].set_title(f"pred: {classes[pr_.argmax()]} ({pr_.max():.2f})"); ax[1, k].axis("off")
plt.show()

In [ ]:
#@title 11. Use the trained model on new images
NEW_DIR = "/content/drive/MyDrive/new_cells"   # put unlabeled images here
import csv
rows = []
for p in sorted(glob.glob(os.path.join(NEW_DIR, "**", "*"), recursive=True)):
    if p.lower().endswith(EXTS):
        pr_ = predict_image(p)
        rows.append([os.path.basename(p), classes[pr_.argmax()]] + [f"{v:.3f}" for v in pr_])
        print(rows[-1])
with open(f"{OUT_DIR}/predictions.csv", "w", newline="") as f:
    csv.writer(f).writerows([["file", "prediction"] + [f"p_{c}" for c in classes]] + rows)
print("Saved", f"{OUT_DIR}/predictions.csv")